<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_MASTER_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS 2.0 — Master pipeline

**One notebook for the whole workflow:** environment → data acquisition → native grid → DEM inventory → geological EDA repair → segment attributes → spatial splits → feature store → baselines → research experiments → discovery evaluation → ensembling → full inference → submission → reports.

Every stage **checks Drive first, reuses valid outputs, builds only what is missing, saves to Google Drive immediately and records provenance**. Re-running any cell after a disconnect resumes where it stopped.

### First time here? Run in three steps

1. **`PREPARE_DATA`** (recommended first run, CPU is enough): reuses the raw files and the saved spatial splits on Drive, rebuilds missing Phase‑1 EDA dependencies, creates the segment attributes and the baseline features, saves everything to Drive and **stops before any model training**.
2. **`BASELINES`** with `train_baselines=['G1']` (CPU): trains / verifies the G1 LightGBM, geographic CV + pseudo-discovery, OOF maps on Drive.
3. **`ADVANCED_RESEARCH`** (GPU runtime): research experiments.

Cells appear in dependency order (`pipe.graph()` is the source of truth):

```
bootstrap → raw_data → native_grid ─┬─ dem_inventory ─────────────────────────────→ dem_pilot (needs feature_store)
                                    ├─ geological_eda → segment_attributes ┄┄┐
                                    ├─ spatial_folds ────────────────────────┼─→ feature_store ─┬─→ dem_pilot
                                    └─ discovery_splits ─────────────────────┘                  │
   spatial_folds + discovery_splits (┄ feature_store, ┄ segment_attributes) ─→ baselines ─→ discovery_validation ─→ ensemble
   feature_store + baselines ─→ advanced_models (GPU)        feature_store ─→ full_inference (GPU) ─→ submission
   reports (always last)                                      ┄ = soft dependency: ordered first, never blocking
```

| decision | meaning |
|---|---|
| `SKIP_CACHED` | outputs exist, pass content checks, and were made under this configuration (or are pre-existing artifacts, adopted with provenance) |
| `BUILD` / `RESUME` | outputs missing / partial: built, or resumed from completed units and checkpoints |
| `INCOMPATIBLE` | outputs exist but differ from the configuration: **never overwritten**; changed protocols get a new versioned folder |
| `BLOCKED` | a prerequisite is missing: the message names the **root failure** and the recovery action |
| `DISABLED` | switched off in `CONFIG`, or a GPU stage on a CPU runtime |

**Failures:** with `failure_policy='interactive'` (default here) a failed critical stage prints the full traceback, the file and line, and the recovery action, then **stops “Run all”**. Completed artifacts are kept. `pipe.explain('<stage>')` shows it again later, also from an earlier session. With `'batch'`, independent stages continue and one grouped report lists each root failure with the stages it blocks.

**Safety:** GPU training, DEM tile downloads, full inference and submission export are off until enabled below. Nothing is ever deleted on Drive, nothing is uploaded to the competition, no URL or token is written to disk or to the Git remote. Pseudo-discovery scores are local-protocol numbers, never leaderboard scores.

Guide: `docs/MASTER_COLAB_GUIDE.md` · dependencies: `docs/PIPELINE_DEPENDENCIES.md` · recovery: `docs/ARTIFACT_RECOVERY.md`.

## 1. Configuration (control panel)

Edit, then run the cells top to bottom (or use the one-shot run in section 6). Keys are validated: a typo raises an error naming the closest valid key.

**Modes:** `SETUP_ONLY` (verify environment and existing data, no downloads) · **`PREPARE_DATA`** (recommended first run: data, EDA repair, segments, splits, features; no training) · `BASELINES` (+ restore / train the baselines listed in `train_baselines`, scoreboard) · `ADVANCED_RESEARCH` (+ GPU research experiments) · `FULL_PIPELINE` (every enabled stage) · `RESUME` (continue from saved state) · `SUBMISSION_ONLY` (promoted model → GeoTIFF; prerequisites validated, not rebuilt).

`ref` is either a **branch** (follows `origin/<branch>`, fast-forward only) or a **commit SHA / tag** (pinned, detached). The code cell prints which one was checked out.

**Execution mode vs experiment fidelity.** `run_mode` decides *which stages* run. `experiment_profile` (`SCREEN` / `VALIDATE` / `FULL`) decides *how much evidence* each GPU research run produces. They are independent keys: neither changes the meaning of the other. See `docs/EXPERIMENT_FIDELITY_AND_COST.md`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                                   # branch name, or a commit SHA / tag to pin the code
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'run_mode': 'ADVANCED_RESEARCH',                 # WHICH STAGES run: PREPARE_DATA -> BASELINES -> ADVANCED_RESEARCH
    'failure_policy': 'interactive',                 # 'interactive': stop on a critical failure | 'batch': continue
    'download_missing': True,                        # only files absent / invalid on Drive AND locally
    'rebuild_missing': True,

    'run_eda': True,                                 # minimal Phase-1 rebuild if outputs/eda is missing (CPU)
    'run_baselines': True,                           # scan / verify baselines (cheap)
    'train_baselines': [],                           # BASELINES step: ['G1'] (CPU); later 'G3', 'B0_lineament'; GPU: 'CNN_A_dti', ...
    'run_advanced_features': True,                   # Phase-4 RS features used by the baselines
    'run_advanced_models': True,                    # GPU research experiments (opt-in)
    # HOW MUCH EVIDENCE each research run produces (independent of run_mode):
    #   'SCREEN'   1 fold, no discovery, <= 5 epochs x 80 steps, proxy metrics on 25 % of the held-out fold (~5 GPU min)
    #   'VALIDATE' 3 folds + discovery split 0, <= 10 epochs x 150 steps, proxy metrics (~30 GPU min)
    #   'FULL'     committed YAMLs unchanged, section-17 evaluation, the only promotion-eligible level (~60 GPU min)
    # Completed FULL runs are reused, never retrained for a cheaper profile; B0 is added as the matched control.
    'experiment_profile': 'SCREEN',
    'advanced_experiments': ['b0_reference', 'b1_bg_weight', 'b2_nnpu_p010', 'c1_geometry_heads'],
    # 'e1_gsn_parity' is a parity check (expected identical to B0), not a candidate: add it only to confirm parity
    'fidelity_overrides': {},                        # e.g. {'SCREEN': {'max_epochs': 4, 'validation_fraction': 0.5}}
    'auto_provision_maps': True,                     # build B0_lineament (label-free, CPU) when B1 needs it
    'checkpoint': {'policy': 'best_and_latest', 'save_every_epochs': 5, 'sync_every_minutes': 15},
    'feature_stage': 'lazy',                         # copy feature caches from Drive on first read ('all' = up front)
    'run_dem_pilot': False,
    'run_scoreboard': True,
    'run_ensemble': True,
    'run_full_inference': False,                     # needs a PROMOTED model (manual decision)
    'run_submission': False,                         # writes GeoTIFFs; never uploads

    'use_gpu': True,                                 # GPU stages are DISABLED automatically when no CUDA GPU is attached
    'max_stage_minutes': 0,                          # e.g. 200 on free tier: clean, resumable stop
    'save_figures': True,
    'save_intermediate': True,
    'seed': 2026,
}

DEM_CONFIG = {
    'enabled': False,
    'mode': 'pilot',                                 # 'pilot' | 'inventory_only' (never the full collection)
    'download': False,                               # explicit approval, after reading the storage estimate
    'pilot_tiles': 24,                               # spatially balanced pilot (f1_dem_pilot.yaml)
    'max_tiles': 30,
    'max_gib': 15.0,
    'download_only_intersecting': True,
    'compute_morphology': True,                      # 1 m morphology before aggregation
    'aggregate_to_100m': True,
}
PRIVATE_REPO = False                                 # True only for a private fork: uses Colab Secret GITHUB_TOKEN per command

## 2–4. Google Drive, GitHub installation, package import

Idempotent: mounts Drive, clones (or updates) the repository at `CONFIG['ref']`, installs `gems-fault-discovery` in editable mode.

* The repository is public, so the clone is **token-free**. Older notebook versions put a token into the remote URL. This cell resets the remote to the plain URL.
* For a private fork set `PRIVATE_REPO = True`. The token from Colab *Secrets* (`GITHUB_TOKEN`) is passed to `git` through environment variables for that one command. It is never part of the URL, `.git/config`, the command line or any output.
* Git failures are **not** ignored. If a branch has diverged or the checkout has local edits, the cell stops with git's message.
* If the code changed after `gems_bench` was already imported in this kernel, the cell asks you to restart the session, so stale code is never used.
* PyTorch is **not** reinstalled (Colab's CUDA build is the right one).

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"        # token-free remote (public repository)

def _git_env():
    """Private fork only: HTTP auth header via GIT_CONFIG_* env vars (git >= 2.31), never in URL / argv / config."""
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    if not tok:
        raise RuntimeError('PRIVATE_REPO=True but the Colab Secret GITHUB_TOKEN is empty / not shared with this notebook')
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:                              # never silently ignored
        raise RuntimeError(f"git {' '.join(args[:3])} failed (exit {r.returncode}):\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)        # drops a token embedded by older notebook versions
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
dirty = git('-C', CODE, 'status', '--porcelain', '--untracked-files=no').stdout.strip()
if dirty:
    print('WARNING: local edits in the checkout (they are kept; a checkout conflict will stop this cell):\n' + dirty)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')   # fails loudly if the branch diverged
    CODE_REF = f'branch {ref} (fast-forwarded to origin/{ref})'
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)             # commit SHA or tag: pinned
    CODE_REF = f'pinned {ref} (detached)'
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)                          # editable install is not visible to a running kernel
os.environ['GEMS_CODE_COMMIT'] = HEAD
print(f'code: {CODE_REF} @ {HEAD}')
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('The code changed while gems_bench was already imported in this kernel. '
                       'Runtime -> Restart session, then run the notebook again from section 1 (Drive stays mounted).')

### Fresh download URLs (only when a raw file is missing on Drive)

DrivenData links are personal and **expire**. Preferred: Colab *Secrets* (key icon, enable notebook access) named `GEMS_URL_FEATURES`, `GEMS_URL_FAULTS`, `GEMS_URL_SUBMISSION` (and optionally `GEMS_URL_DEM_JSON`). Alternative: set `ASK_FOR_URLS = True` to paste them here (hidden input). URLs live only in memory; the acquisition manifest stores only the host and a hash. Files already valid on Drive are never downloaded again.

In [ ]:
URLS = {}
ASK_FOR_URLS = False
if ASK_FOR_URLS:
    from getpass import getpass
    for key in ('features', 'faults', 'submission', 'dem_json'):
        u = getpass(f'URL for {key} (Enter to skip): ').strip()
        if u:
            URLS[key] = u

from IPython.display import display
from gems_bench.pipeline import Pipeline
pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG, urls=URLS)
display(pipe.cfg.table())

## 5. Environment and GPU verification

`torch.cuda.is_available()` is what counts. A CPU-only PyTorch build, or a CPU runtime, is not GPU support. Without CUDA:

* **CPU work stays available**: raw data, EDA repair, segments, splits, feature store, LightGBM baselines (G1, G3, RS05, B0).
* **GPU stages are DISABLED** (`advanced_models`, `full_inference`), and requested GPU baselines (`CNN_*`, `P4_B_gated`) are reported as *not attempted on this runtime*. Nothing is faked, and no model is allocated.
* To train on a GPU, use *Runtime → Change runtime type → GPU (T4 / L4 / A100)*, reconnect, and run again from section 1. Do **not** reinstall torch: Colab's preinstalled build is CUDA-enabled once a GPU is attached.

When a GPU is present, batch size, precision (bf16 only on compute capability ≥ 8) and inference batch adapt to it. The effective batch, patch size and epochs never change.

In [ ]:
pipe.step('bootstrap')
from gems_bench.training import gpu
g = pipe.pc.gpu()
cuda_ok, cuda_why = pipe.pc.cuda_ready()
print({k: g.get(k) for k in ('cuda', 'name', 'vram_gb', 'capability', 'torch', 'torch_cuda')})
if cuda_ok:
    print('autoconfig (256 px, effective batch 16, 29 channels):',
          gpu.autoconfig(g, patch_px=256, effective_batch=16, n_channels=29))
else:
    print(f'CPU runtime: {cuda_why}')
    print('CPU preprocessing and LightGBM baselines run normally; GPU stages are DISABLED (not failed).')
display(__import__('pandas').read_csv('/content/gems_work/reports/pipeline/environment_report.csv'))

## 6. First-run overview: storage, raw data, GPU and execution plan

Read-only. `pipe.overview()` prints the configuration, the checked-out commit, Drive, GPU, raw-data status and the planned action per stage, validated against what is **on Drive**. Nothing is built here. Expected on the current Drive in `PREPARE_DATA`: `raw_data`, `native_grid`, `spatial_folds` and `discovery_splits` are `SKIP_CACHED` (no download, no regeneration of folds), while `geological_eda`, `segment_attributes` and `feature_store` are `BUILD` / `RESUME`.

**One-shot alternative:** `pipe.run()` executes the whole configured mode in dependency order. The cells below run the same stages one section at a time, with displays.

In [ ]:
plan = pipe.overview()
display(plan)
display(pipe.graph())
from gems_bench.pipeline.status import storage
display(storage(pipe.pc))

RUN_EVERYTHING_NOW = False                           # True = run the configured mode in one call
if RUN_EVERYTHING_NOW:
    display(pipe.run())

## 7. Raw data inventory and one-time download

Per file: Drive (canonical path, then accepted alternates such as `data/raw/example_submission.tif`) → local disk → resumable download (`.partial`, HTTP Range, retries) → GeoTIFF / JSON validation → verified atomic copy to Drive → manifest. Downloads happen only for files that are missing or invalid in both places.

In [ ]:
from gems_bench.data.acquisition import inventory
display(inventory({'drive': pipe.pc.drive, 'local': pipe.pc.local}))
pipe.step('raw_data')

## 8. Native geospatial data verification

100 m GeoDAWN grid (EPSG:32611), mapped-fault labels (value ≥ 1), valid footprint, submission template on the same grid. Recorded in `environment/pipeline/native_grid.json` with raw-file fingerprints, so a new data release is detected instead of silently mixed. Every later stage (DEM inventory, EDA, splits, features) depends on this one.

In [ ]:
pipe.step('native_grid')
display(pipe.pc.data.summary())

## 9. USGS 1 m DEM inventory (metadata only) and optional pilot

Runs after the native grid (it needs the study-area footprint). The inventory parses the URL JSON and identifies source projects, duplicate URLs, overlapping projects per tile, study-area intersection and the storage estimate. It also writes a footprint GeoPackage and a figure. `data/raw/tnm_items.json` is accepted and copied to `data/raw/dem_1m/tnm_urls.json`. **No tile is downloaded here.** The pilot (after the feature store, section 12) selects a spatially balanced subset and downloads it only with `DEM_CONFIG['download'] = True`. The DEM is optional: nothing else waits for it.

In [ ]:
pipe.step('dem_inventory')
import json, pathlib
est = pathlib.Path('/content/gems_work/data/raw/dem_1m/inventory/dem_storage_estimate.json')
if est.exists():
    print(json.dumps(json.loads(est.read_text()), indent=1))

## 10. Phase 1 EDA dependency repair, then native segment attributes

Only the EDA artifacts later phases actually read are regenerated: fault segments + archetypes, fused lineament maps and cross-modal orientation. They are built with the same `gems_eda` functions and defaults as notebook 01 (2048 px working grid, seed 7), in `outputs/eda_rebuild/<version>/`. This takes about 15–45 min of CPU, once.

* Each artifact group is installed into `outputs/eda/` **only if the whole group is absent**. Historical files are never mixed or overwritten. Installed files are listed in `manifest.json` with `origin: rebuilt`.
* **Resume:** a finished rebuild folder (`REBUILD_COMPLETE.json`) is reused instead of recomputed. An install interrupted half-way (`outputs/eda/INSTALL_STATE.json` = `in_progress`) is completed from the same rebuild folder.
* `segment_attributes` then writes `outputs/benchmark/datasets/native_segment_ids.tif` (native 100 m grid, `seg_id` band) and `segment_attributes.parquet`. `native_segments_provenance.json` records the sha256 of the source segment set, so new segment ids are never paired with an old raster.

If this cell fails, the message shows the file:line and the recovery. `pipe.explain('geological_eda')` repeats it.

In [ ]:
pipe.step('geological_eda')
pipe.step('segment_attributes')
import pandas as pd
p = pathlib.Path('/content/gems_work/outputs/eda/manifest.json')
if p.exists():
    display(pd.DataFrame(json.loads(p.read_text())).T)
for f in ('outputs/eda/INSTALL_STATE.json', 'outputs/benchmark/datasets/native_segments_provenance.json'):
    q = pipe.pc.local / f
    if q.exists():
        print(f, '->', json.loads(q.read_text()))

## 11. Phase 2 geographic folds and pseudo-discovery splits

Protocol: 3 geographic folds, 20 km blocks, 13 km buffer · 3 discovery splits, 1 km fault-system linking · seed 2026. The folders already on Drive (`outputs/benchmark/folds/geo_k3_b20km_buf13km_s2026`, `discovery_d3_link1km_s2026`) are **reused** after content checks: grid, band count, roles, recorded parameters. They are never regenerated because an experiment starts. Only a changed protocol produces a new, differently named folder.

In [ ]:
pipe.step('spatial_folds')
pipe.step('discovery_splits')
print(pipe.pc.fold_rel, '|', pipe.pc.discovery_rel)
from gems_bench.pipeline.status import fold_figure, save_fig
fig = fold_figure(pipe.pc)
if fig is not None:
    display(fig); save_fig(pipe.pc, fig, 'validation_design')

## 12. Feature store: create once, reuse

Families: raw 19-channel geophysics · Phase 2 engineered · Phase 1 `eda:*` maps · Phase 4 remote-sensing structural features (those the baselines use, or all) · Phase 5 fault-normal profiles · 1 m DEM aggregates. Only missing features are computed. Each family is pushed to Drive as soon as it exists.

Feature-group prerequisites: only the `eda:*` family needs the Phase-1 EDA rasters. If they are missing, that family is deferred and named in the error, while every other family is still built and kept. This means a G1 LightGBM baseline never waits for the EDA, the advanced feature bank or the DEM pilot. `data/feature_store/pipeline_feature_registry.csv` declares, per feature: name, source, transform, scale, support, CRS / grid, NoData behaviour, standardisation and a spec hash.

In [ ]:
pipe.step('feature_store')
reg = pathlib.Path('/content/gems_work/data/feature_store/pipeline_feature_registry.csv')
if reg.exists():
    fr = pd.read_csv(reg)
    display(fr.groupby('family').agg(n=('name', 'size'), cached=('cached', 'sum')))

pipe.step('dem_pilot')                               # only runs when the DEM pilot is enabled

if CONFIG['run_mode'] == 'PREPARE_DATA':
    print('PREPARE_DATA ends here (no model training). Next: set run_mode="BASELINES", train_baselines=["G1"], '
          'and run from section 1. The baseline cells below only report the cached state in this mode.')
    pipe.failure_report()

## 13. Baselines: restore or train

G1 terrain LightGBM, G3 engineered LightGBM, RS05 (Phase 4), CNN-A and CNN-B (distance-aware), gated dual encoder, plus the label-free B0 lineament floor. A reported score is not a model: each baseline is `AVAILABLE` only if its finished run folder and maps are found and verified. Otherwise it is `NOT_AVAILABLE`, or it is retrained from the committed configuration snapshot **only when listed in `train_baselines`** and the mode is `BASELINES` or later.

**Next execution target: G1** (`run_mode='BASELINES'`, `train_baselines=['G1']`, CPU). It runs geographic CV, then a full fit per pseudo-discovery split. OOF and discovery maps and `PROVENANCE.json` (fold / split sha256, `historical_equivalence: false`) go to Drive. A regenerated G1 is a new run under the historical recipe. It is not a reproduction of the historical DTI until its metrics have been checked. GPU baselines on a CPU runtime are listed as *not attempted*.

In [ ]:
pipe.step('baselines')
display(pipe.baselines())

## 14–15. Research experiments: inventory → reuse → SCREEN / VALIDATE / FULL

**Progressive experimentation.** Cheap screening first, full validation only for shortlisted candidates:

`SCREEN` (1 fold, capped budget, proxy metrics) → shortlist → `VALIDATE` (3 folds + 1 discovery split) → shortlist → `FULL` (committed protocol, section-17 gate).

* **Reuse first.** `pipe.inventory()` lists every experiment folder on Drive (profile, completed folds / splits, metrics, reusability, next action). `pipe.plan_experiments()` shows what this session will do: `REUSE_FULL` (a compatible FULL run is complete: never retrained), `DONE`, `RESUME` (unit-level, from checkpoints), `RUN`, `NOT_NEEDED`.
* **Identity.** SCREEN / VALIDATE runs live in their own folders (`<name>__screen__<hash>`), so they can never overwrite or resume a FULL run. FULL keeps the historical ids (`b0_reference__d54e938a`, ...).
* **Matched control.** Every SCREEN / VALIDATE candidate is compared with `b0_reference` trained under the *same* profile, on the *same* fold, budget and scored pixels (mask digests are checked).
* **Proxy metrics.** SCREEN / VALIDATE numbers are labelled `proxy_*`. They never enter the scoreboard, the registry or the promotion gate. Only FULL runs do.
* **B1** needs the label-free `B0_lineament` map. It is found and verified on Drive, or built with its original CPU recipe (minutes) before B1 starts.
* **E1** (`e1_gsn_parity`) is a *parity check*. It uses the same modules, widths, construction order and seed as B0, so identical training is the expected result, not an architectural gain.

In [ ]:
# P0 - read-only: what already exists (no GPU)
inv = pipe.inventory()
display(inv[['experiment_id', 'role', 'profile', 'folds_done', 'discovery_done', 'evaluation_done',
             'metrics_available', 'reusable', 'next_action']])
res = pipe.compare_results()          # reports/experiment_screening: metrics, paired deltas, shortlist, figures
print('report:', res['dir'])

# what this session will train at the selected profile (read-only)
display(pipe.plan_experiments())

# train only the missing work (RUN / RESUME); completed FULL runs are reused
pipe.step('advanced_models')

### Research dashboard and promotion to the next fidelity level

The dashboard shows the session profile, completed / incomplete experiments, remaining folds and splits, estimated GPU minutes, recorded GPU time, available maps, shortlist decisions and Drive usage (CSV + PNG/SVG under `reports/pipeline`).

Shortlist decisions (`PROMOTE_TO_VALIDATE`, `PROMOTE_TO_FULL`, `REVIEW`, `REJECT`, `PARITY_CONFIRMED`, `INCOMPARABLE`) are **recommendations** from configurable triage heuristics (`screening_rules`). Moving to the next level is your decision: copy the shortlisted names into `advanced_experiments`, set `experiment_profile` and re-run this section. Promoting a model for **submission** remains a separate, manual registry transition (section 16–17).

In [ ]:
dash = pipe.research_dashboard()

next_validate = pipe.shortlist('SCREEN')        # recommended for VALIDATE
next_full = pipe.shortlist('VALIDATE')          # recommended for FULL
print('SCREEN -> VALIDATE:', next_validate)
print('VALIDATE -> FULL  :', next_full)
# To promote (explicit): CONFIG.update({'experiment_profile': 'VALIDATE', 'advanced_experiments': next_validate})
# then re-run the configuration and pipeline cells, and this section.

## 16–17. Geographic and pseudo-discovery validation, model comparison and promotion

One protocol for every available model: geographic DTI (cross-fitted area budget) and pseudo-discovery DTI (withheld systems excluded / retained as FP) are reported **separately**; `public_lb` comes only from your manual `reports/leaderboard_log.csv`. Promotion to `promoted` is a manual, evidence-backed registry transition.

In [ ]:
pipe.step('discovery_validation')
sb = pathlib.Path('/content/gems_work/reports/pipeline/scoreboard_latest.csv')
if sb.exists():
    b = __import__('pandas').read_csv(sb)
    display(b[[c for c in ('model', 'geo_dti_area', 'geo_fold_min', 'disc_dti_excl', 'disc_dti_retained_fp',
                           'short_recall_w', 'public_lb') if c in b.columns]])
from gems_bench.artifacts import ModelRegistry
display(ModelRegistry(pipe.pc.local / 'model_registry').table())
# Manual promotion (only when the candidate's promotion_decisions say promote=True):
# from gems_bench.experiment.promote import promote
# promote('<model_name>', '<experiment_id>'); pipe.push_all()

## 18. Ensemble optimisation

Members (≥ 2 with OOF + every discovery map) enter as per-fold rank quantiles; geographic weights are chosen on other folds, discovery weights with each split's withheld systems removed. An ensemble must pass the same gate against its best member.

In [ ]:
pipe.step('ensemble')

## 18b. M6 — model selection from existing FULL experiments (no training)

**Central question.** Can B0's short-fault recall and retained-FP robustness be combined with B1's geographic generalisation and discovery DTI? M6 answers it from the **saved FULL predictions only**. It uses no GPU and does no neural training, and it only reads the original experiment folders.

**Prerequisites.** Run the configuration, Drive and install cells (1–4) and sections 7–12; these are cached and do not train. M6 stages only the prediction rasters, manifests and metrics of the analysed experiments, never their checkpoints.

| phase | what | leakage protection |
|---|---|---|
| M6.1 audit | inventory, provenance, compatibility, missing artifacts (`COMPLETE / PARTIAL / INCOMPATIBLE / MISSING / INVALID`) | read-only; stops if a FULL map is missing |
| M6.2 paired | one consolidated section-17 scorecard; per-fold / per-split deltas vs B0 and B1; spatial-block bootstrap | area budget of fold *k* chosen on the other folds |
| M6.3 calibration | raw, rank, robust quantile, linear, temperature, isotonic, area budget, absolute-threshold transfer | every label-dependent parameter cross-fitted |
| M6.4 ensembles | B0+B1 mean / rank mean / nested weighted rank and robust; then B2 / C1 additions | weights + budget per fold chosen on the other folds; discovery weights never see the scored split |
| M6.5 complementarity | equal-area overlap; both / B0-only / B1-only / neither by length, fold, strike, contrast, junctions; case-study panels | descriptive |
| M6.6 postprocessing | small-component filter, gap closing, direction-aware linking, skeleton width, minimum length, orientation filter | parameters chosen on the other folds; identity always allowed |
| M6.7 gate | section-17 gate extended, against **both** B0 and B1: `PROMOTE / REVIEW / REJECT / INSUFFICIENT_EVIDENCE` | FULL members only; nothing is promoted automatically |

Every analysis unit is fingerprinted, so a re-run reuses finished units (`overwrite_existing=True` recomputes them). Outputs go to `/content/drive/MyDrive/GEMS_2/reports/m6/`: tables, figures with their CSV data, and `M6_RESEARCH_REPORT.md`. Guide: `docs/M6_RESULTS_GUIDE.md`. **No improvement is claimed until these cells have run.**

In [ ]:
M6_CONFIG = {                                       # overrides configs/m6/model_selection.yaml (unknown keys rejected)
    'enabled': True,
    'mode': 'ANALYZE_EXISTING',                      # the only mode: M6 never trains
    'models': ['b0_reference', 'b1_bg_weight', 'b2_nnpu_p010', 'c1_geometry_heads'],
    'primary_pair': ['b0_reference', 'b1_bg_weight'],
    'run_calibration': True,
    'run_ensemble': True,
    'run_complementarity': True,
    'run_postprocessing': True,
    'run_promotion_audit': True,
    'train_models': False,                           # True is refused
    'overwrite_existing': False,                     # True recomputes every analysis unit
    'save_figures': True,
    'save_csv': True,
    'save_parquet': True,
    # e.g. 'ensemble': {'weights': [0.0, 0.25, 0.5, 0.75, 1.0], 'refine': True}, 'save_rasters': 'all'
}

from IPython.display import Image, Markdown, display

m6 = pipe.m6(M6_CONFIG)

def show(names, sub='figures'):
    for n in names:
        p = m6.ws.root / sub / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', p.name)

# M6.1 - read-only audit; raises M6ArtifactError (with a recovery hint) if a required FULL map is missing
if M6_CONFIG['enabled']:
    audit = m6.audit()
    inv = audit['inventory']
    display(inv[[c for c in ('model', 'kind', 'status', 'experiment_id', 'profile', 'folds_done', 'n_disc_maps',
                             'feature_version', 'git_commit', 'registry_status', 'reasons') if c in inv]])
    display(audit['compatibility'])
    display(audit['missing'])

### M6.2 Consolidated comparison and paired geographic evidence

All models are re-scored together on one protocol (same folds, splits, budgets and policies). `reference_check` compares the numbers from the previous review with the persisted and recomputed scores and **reports** any difference. With three folds and three splits the paired tables are descriptive: direction, size, worst unit, and a spatial-block bootstrap (the folds' 20 km blocks, resampled inside each held-out zone). They are not significance tests.

In [ ]:
if M6_CONFIG['enabled']:
    m6.scorecard()
    paired = m6.paired()
    display(paired['reference_check'])               # previous review's numbers vs persisted / recomputed (never reconciled)
    mc = paired['model_comparison']
    display(mc[[c for c in ('model', 'geo_dti_area', 'geo_fold_min', 'geo_fold_std', 'disc_dti_excl',
                            'disc_dti_retained_fp', 'short_recall_w', 'geo_far_share_at_match') if c in mc]])
    display(paired['fold_metrics'].pivot_table(index='model', columns='fold', values='dti_heldout'))
    if len(paired['discovery_metrics']):
        display(paired['discovery_metrics'].pivot_table(index=['policy', 'model'], columns='split', values='dti_heldout'))
    display(paired['paired_summary'])
    display(paired['spatial_bootstrap'])
    show(['model_comparison', 'geo_dti_per_fold', 'paired_dti_differences', 'discovery_dti_per_split',
          'worst_fold_comparison', 'short_fault_recall_comparison', 'score_sensitivity'])

### M6.3 Area-budget and score calibration

For a fixed truth set, a binary per-fold top-*a* mask maximises DTI, so monotone recalibration cannot change it. `calibration_invariance` checks that raw and rank-normalised scores give identical masks. What can differ is how well a budget **transfers** between regions: `threshold_transfer` compares an absolute score cut with the per-fold area fraction, both chosen on the other folds.

In [ ]:
if M6_CONFIG['enabled'] and M6_CONFIG['run_calibration']:
    cal = m6.calibration()
    display(cal['calibration_results'][['model', 'family', 'label_free', 'dti_crossfit', 'dti_insample', 'optimism',
                                        'fold_min', 'choice_on_grid_edge']])
    display(cal['calibration_invariance'])
    display(cal['threshold_transfer'])
    show(['area_budget_curves', 'calibration_summary'])

### M6.4 Ensemble optimisation (nested)

B0 + B1 first: mean, rank mean, and weighted rank / robust blends *S = α R(S_B0) + (1 − α) R(S_B1)* with α on `[0, 0.25, 0.5, 0.75, 1]`. For held-out fold *k*, α and the area budget are chosen on the other folds, and refined around fold *k*'s own dev choice when it is interior. Discovery weights are chosen with the scored split's withheld systems removed (`geo_masked`) or on the other splits (`loso`). B2 / C1 are added only afterwards and must beat the best pair (`extension_justified`).

In [ ]:
if M6_CONFIG['enabled'] and M6_CONFIG['run_ensemble']:
    ens = m6.ensembles()
    ec = ens['candidates']
    b0, b1 = m6.pair
    display(ec[[c for c in ('candidate', 'members', 'method', 'normalisation', 'discovery_weights', 'geo_dti',
                            'geo_fold_min', 'disc_dti_excl', 'disc_dti_retained_fp', 'short_recall_w',
                            f'delta_geo_dti_vs_{b0}', f'delta_geo_dti_vs_{b1}', f'delta_disc_dti_excl_vs_{b0}',
                            f'delta_disc_dti_excl_vs_{b1}', 'geo_weights_per_fold', 'relative_inference_cost',
                            'extension_justified') if c in ec]])
    display(m6.ws.load_table('ensemble_index_audit'))      # selection zones never include the scored zone
    display(m6.ws.load_table('ensemble_selection_discovery'))
    display(ens['bootstrap'])
    show(['score_vs_cost', 'ensemble_weight_choices'])

### M6.5 Complementarity and geological errors

At identical area budgets: score correlation, IoU and disagreement area. Mapped segments are classed as recovered by both / B0 only / B1 only / neither, by length, fold, strike, geophysical contrast, junction context and distance to the fold edge. Withheld systems are classed the same way, and false-positive exposure is shown by distance to mapped faults. Case-study panels (geophysics, mapped faults, B0, B1, ensemble, agreement) share extent and scale and are regenerated from `case_study_windows.csv`. Agreement far from mapped faults is an **unmapped prediction**, not a confirmed fault.

In [ ]:
if M6_CONFIG['enabled'] and M6_CONFIG['run_complementarity']:
    comp = m6.complementarity()
    pcm = comp['model_complementarity']
    display(pcm[pcm.zone.astype(str) == 'all'])
    fs = comp['fault_system_summary']
    display(fs[fs.stratum == 'length_bin'].pivot_table(index='value', columns='class', values='trace_km', aggfunc='sum'))
    display(fs[fs.stratum == 'fold'].pivot_table(index='value', columns='class', values='trace_km', aggfunc='sum'))
    if len(comp['discovery_system_classes']):
        display(comp['discovery_system_classes'].groupby(['split', 'class']).size().unstack(fill_value=0))
    display(comp['error_attribution'])
    show(['model_disagreement', 'short_fault_analysis'])
    for w in comp['case_study_windows'].head(8).to_dict('records'):
        show([f"{w['kind']}_{w['source']}_{w['id']}"], sub='figures/geological_case_studies')

### M6.6 Topology-aware postprocessing (optional research layer)

Each operation is evaluated alone, with its parameter and area budget chosen on the other folds. The identity is always a candidate. Faults can be short, discontinuous, branched and curved, so every row shows the change in < 2 km recall, fragmentation, junctions and far-field share next to the DTI change. Operations are combined only if each one helped alone without costing short-fault recall.

In [ ]:
if M6_CONFIG['enabled'] and M6_CONFIG['run_postprocessing']:
    post = m6.postprocessing()
    pa = post['ablation']
    display(pa[[c for c in ('model', 'operation', 'dti_crossfit', 'delta_vs_identity', 'short_recall_w',
                            'delta_short_recall_w', 'n_components', 'delta_fragmentation_per_100km',
                            'delta_n_junctions', 'delta_far_share', 'n_folds_applied') if c in pa]])
    show(['postprocessing_ablation'])

### M6.7 Selection gate and research report

Section-17 criteria plus short-fault recall, area calibration, spatial stability, leakage, provenance and cost, assessed against **both** B0 and B1. A candidate is recommended only with `PROMOTE` against both. Otherwise the report says that no candidate passes. Registry promotion and submission stay manual (sections 16–17 and 19–20).

In [ ]:
if M6_CONFIG['enabled']:
    if M6_CONFIG['run_promotion_audit']:
        g = m6.gate()
        d = g['decisions']
        display(d[d.control == 'ALL'][['candidate', 'verdict', 'reason']])
        display(d[d.control != 'ALL'])
        print('recommendation:', g['recommendation'])
    report = m6.report()
    display(m6.summary())                            # compact end-of-section summary
    print('report:', report)
    display(Markdown(report.read_text(encoding='utf-8')[:6000]))

## 19–20. Full-area inference and competition GeoTIFF

Uses the registry **champion** (status `promoted`): fold models averaged over the full footprint (chunked, Hann-blended tiles), area-budget export policy, one validated single-band float32 GeoTIFF per known-fault policy in `submissions/` with provenance JSON. **Uploading is manual**; record each upload in `reports/leaderboard_log.csv`.

In [ ]:
pipe.step('full_inference')
pipe.step('submission')
import pandas as pd
from gems_bench.evaluation.scoreboard import LEADERBOARD_COLUMNS
log = pipe.pc.local / 'reports' / 'leaderboard_log.csv'
if not log.exists():
    pd.DataFrame(columns=LEADERBOARD_COLUMNS).to_csv(log, index=False)
display(pd.read_csv(log))

## 21. Figures and research reporting

`reports/pipeline/` on Drive: `PIPELINE_REPORT.md`, `stage_status.csv`, `dashboard_summary.csv`, `baseline_status.csv`, `experiment_status.csv`, `scoreboard_latest.csv`, `figures/*.png|svg`. Per-experiment tables and figures stay in `experiments/<id>/`.

In [ ]:
pipe.step('reports')
pipe.dashboard()

## 22. Final artifact and manifest validation

Re-validates every stage's outputs **on Drive** (what survives the session), lists the stage records and the latest events.

In [ ]:
pipe.push_all()
final = pipe.plan(mode='FULL_PIPELINE', root='drive')
display(final[['stage', 'outputs', 'planned', 'last_status', 'last_origin', 'last_finished', 'detail']])
display(pipe.status())
display(pipe.events(30))
for s in ('feature_store', 'baselines', 'discovery_validation', 'submission'):
    print(s, '->', pipe.outputs(s)[:3])

## 23. Resume, inspect a failure, update the code

After a disconnect or in a new session: **run sections 1–4 again, then the same cells (or `pipe.run()`)**.

* Finished stages are `SKIP_CACHED` (validated on Drive, staged back to `/content/gems_work` only when needed). Raw files and folds are never downloaded or regenerated again.
* **A stage failed?** `pipe.explain('<stage>')` prints the error, `file:line`, the recovery action and the traceback, also for a failure recorded in an earlier session. `pipe.failure_report()` lists each root failure and the stages it blocks.
* **After a fix is pushed to GitHub:** *Runtime → Restart session*, then run from section 1. The installation cell fast-forwards `ref` (or checks out the pinned SHA) and prints the commit. Completed stages are skipped; the failed stage resumes.
* Interrupted feature builds continue with the missing features. Interrupted experiments continue at the next CV fold / discovery split and resume training from the last checkpoint pushed to Drive (`ckpt_sync_min`).
* To change a protocol (folds, features, experiment settings), edit the configuration. A new versioned folder / experiment id is created, and earlier results are kept.
* On short sessions set `max_stage_minutes` (e.g. 200): training stops cleanly at an epoch boundary and resumes next time.

The original notebooks `00`–`07` remain as focused research entry points; this notebook is the recommended start.